# agentfix (LangGraph edition) — Google Colab

Browser-only setup for the workshop. No local install, no 8 GB download.

Run the cells **in order**. Sections 1–7 are setup; the exercises start after that.

What is different from a local setup: you edit files and run checks from cells rather than in
an IDE. The code, the exercises and the tests are identical.


## 0. Check the runtime

A CPU runtime is fine — the model is small. No GPU needed.


In [ ]:
!python --version
!free -g | head -2 || true


## 1. Configuration

Change `REPO_URL` if you forked the repository.


In [ ]:
REPO_URL = "https://github.com/jelenadjuric01/agentfix-langchain.git"
MODEL = "qwen2.5-coder:1.5b"   # ~1 GB. Mellum2 is too large for a free Colab runtime.
print(REPO_URL, MODEL)


## 2. Install and start Ollama

The server runs in the background inside this runtime and listens on `localhost:11434`.


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
!curl -s http://localhost:11434/api/tags | head -c 200


## 3. Pull the model

No `ollama create` step is needed. This edition talks to Ollama's native API, which honours a
per-request `num_ctx`, so the context window is set by the client rather than baked into a
derived model.


In [ ]:
import os; os.environ["MODEL"]=MODEL
!ollama pull $MODEL


In [ ]:
# Smoke test: does the server answer at all?
!ollama run $MODEL 'Reply with exactly: ready' --keepalive 5m


## 4. Clone the student repository

`main` is the stubbed starting point. That is deliberate — the exercises live in the gaps.


In [ ]:
%cd /content
!rm -rf agentfix-langchain
!git clone --quiet $REPO_URL agentfix-langchain
%cd /content/agentfix-langchain
!git branch --show-current


## 5. Confirm you are on the stubbed `main`

This cell **should** report failures. If the exercise tests pass here, you are on the wrong
branch and there is nothing to do.


In [ ]:
!grep -n 'EXERCISE(stage-' src/agentfix/agent/graph.py


## 6. Make accidental pushes impossible

You will be committing nothing and pushing nothing, but a stray `git push` in a shared
notebook is worth ruling out.


In [ ]:
!git remote set-url --push origin DISABLED
!git remote -v


## 7. Install the project

Colab already has a Python, so this installs into it directly rather than via `uv`.


In [ ]:
!python -m pip install -q -e '.[dev]'
import os; os.environ["MELLUM_MODEL"] = MODEL
!agentfix doctor || true


`doctor` may report the RAM check differently here than on a laptop, and the model name it
expects comes from `MELLUM_MODEL`, which the cell above set for you. A `generation` PASS means
the whole path works.


# Stage 1 — When is the agent done?

Read the instructions, then edit `src/agentfix/agent/graph.py`.


In [ ]:
!cat exercises/stage_1/README.md


In [ ]:
!grep -n 'EXERCISE(stage-1)' -A 6 src/agentfix/agent/graph.py


### See the failure first

Always worth doing once: the tests describe the job better than any prose can.


In [ ]:
!python -m unittest exercises.stage_1.test_stage_1 -v 2>&1 | tail -30


### Edit the file

In Colab: double-click `src/agentfix/agent/graph.py` in the file browser on the left, edit,
then **Ctrl+S**. The next cell picks up the change with no reinstall — the project is
installed in editable mode.


In [ ]:
!python -m unittest exercises.stage_1.test_stage_1 -v 2>&1 | tail -20


### Stuck? Read the answer without losing your work


In [ ]:
!git --no-pager diff main stage-1-solution -- src/agentfix/agent/graph.py


# Stage 2 — Catching a stuck model


In [ ]:
!cat exercises/stage_2/README.md


In [ ]:
!grep -n 'EXERCISE(stage-2)' -A 6 src/agentfix/agent/graph.py


In [ ]:
!python -m unittest exercises.stage_2.test_stage_2 -v 2>&1 | tail -30


### Stuck?


In [ ]:
!git --no-pager diff stage-1-solution stage-2-solution -- src/agentfix/agent/graph.py


# Both stages done — run the whole thing

First the repo's own suite, then the agent against a real bug.


In [ ]:
!python -m unittest discover -s exercises -t . 2>&1 | tail -5
!python -m unittest discover -s tests -t . 2>&1 | tail -5


In [ ]:
!agentfix solve tasks/workshop/01-shopcart --verbose


A 1.5B model may not solve it. Read the trace either way: one line per model turn, one per
tool call, the context size growing, and `(NO REASONING)` on every turn that acted. That
marker is the finding — this is an Act-only agent, and the sequel to this workshop is about
closing exactly that gap.


## Optional — the harder task


In [ ]:
!agentfix solve tasks/workshop/02-invoice --verbose


## Optional — compare your work against the finished agent


In [ ]:
!git --no-pager diff stage-2-solution -- src/agentfix/agent/graph.py


## Optional — the framework comparison

`agent/prebuilt.py` builds the same agent from `langchain.agents.create_agent` and its
middleware, and documents which of the three invariants that actually buys you.


In [ ]:
!python -m pip install -q -e '.[dev,prebuilt]'
!python -m unittest discover -s tests -p 'test_prebuilt.py' -t . -v 2>&1 | tail -20


## Optional — an eval run

Slow: it runs the model once per task, sequentially, and that is deliberate. Precomputed
results from a Mellum2 machine are in `results/precomputed/` if you would rather read than wait.


In [ ]:
!cat results/precomputed/workshop.json
# !agentfix eval --suite workshop     # uncomment to run it for real
